# DuckLake v1.0 vs Delta-rs + Polars — Benchmark

Jupyter front-end for the benchmark. Edit the `RunSelection` cell to control scope, then
Run All. Same code path as the marimo notebook and the headless CLI runner.

See `docs/superpowers/specs/2026-04-26-benchmark-assumptions.md` for methodology and
limitations.


In [1]:
import sys
from pathlib import Path

from loguru import logger

logger.remove()
logger.add(sys.stderr, level="INFO")

_BENCH_DIR = Path.cwd()
if _BENCH_DIR.name != "benchmark":
    _BENCH_DIR = _BENCH_DIR / "notebooks" / "benchmark"
if str(_BENCH_DIR) not in sys.path:
    sys.path.insert(0, str(_BENCH_DIR))

from helpers import (
    RunSelection,
    export_results,
    load_config,
    pivot_comparison,
    results_to_dataframe,
    run_benchmark,
    speedup_ratios,
    storage_efficiency,
)

CONFIG_PATH = _BENCH_DIR / "config.yaml"
RESULTS_DIR = str(_BENCH_DIR / "results")
config = load_config(CONFIG_PATH)
config.name

'DuckLake v1.0 vs Delta-rs + Polars'

## Run selection

Edit the lists below to choose what to run. Defaults are `tiny` + `small` on local storage,
both engines, all operations. For the conference headline (100 GB, 16 GB RAM), set
`sizes=["xl"]` and pick a small subset of operations.


In [3]:
selection = RunSelection(
    sizes=["tiny", "small"],
    storage_modes=config.default_storage_modes,
    engines=["ducklake", "delta"],
    operations=config.operations.all_operations,
)
selection

RunSelection(sizes=['tiny', 'small'], storage_modes=['local'], engines=['ducklake', 'delta'], operations=['write_append', 'write_overwrite', 'read_full_scan', 'read_filtered_scan', 'read_aggregation', 'merge_upsert'])

In [ ]:
results = run_benchmark(config, selection)
len(results)

In [ ]:
df = results_to_dataframe(results)
df

## Side-by-side comparison


In [ ]:
pivot_comparison(df)

## Speedup ratios

`time_speedup_ducklake_vs_delta` > 1 means DuckLake is faster.


In [ ]:
speedup_ratios(df)

## Storage efficiency

`disk_usage_mb` is data files only. `postgres_metadata_mb` is the DuckLake catalog
delta over the empty baseline; Delta reports 0. `total_storage_mb` is the slide-ready sum.


In [ ]:
storage_efficiency(df)

In [ ]:
if not df.is_empty():
    csv_path = export_results(df, RESULTS_DIR)
    print(f"Results exported to {csv_path}")